# LoRA-Laplace Training Pipeline

This notebook runs the full training and calibration pipeline for the LoRA + diagonal Laplace Approximation uncertainty backend:

1. **Configuration** — paths, model, hyperparameters
2. **Fine-tune** — train LoRA adapters on (source, summary) pairs
3. **Fit Laplace approximation** — compute diagonal Fisher information and posterior variance
4. **Score calibration corpus** — run posterior-sampled forward passes, collect sentence-level uncertainty
5. **Fit quantile normalizer** — map raw epistemic MI scores to a 0–100 display scale
6. **Upload to HuggingFace Hub** — push adapter, sampler and quantile config

**Input**: `data/summaries_v3.jsonl` (produced by `01_data_ingestion.ipynb`)

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path("..")
DATA_DIR      = ROOT / "data"
MODELS_DIR    = ROOT / "models"
CONFIG_DIR    = ROOT / "config"
MODELS_DIR.mkdir(exist_ok=True)

SUMMARIES_FILE    = DATA_DIR / "summaries_v3.jsonl"
ADAPTER_DIR       = MODELS_DIR / "bart-large-xsum-lora"
SCORES_FILE       = DATA_DIR / "uncertainty_scores_lora_laplace.jsonl"
SAMPLER_FILE      = ADAPTER_DIR / "laplace_sampler.npz"
QUANTILES_FILE    = CONFIG_DIR / "uncertainty_quantiles_lora_laplace.json"

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "facebook/bart-large-xsum"

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_RANK           = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS            = 3
BATCH_SIZE        = 4
GRAD_ACCUM        = 4
LEARNING_RATE     = 3e-4
VAL_SPLIT         = 0.1
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
SEED              = 42
BF16              = True   # set False if not on MPS/CUDA with bf16 support

# ── Laplace calibration ───────────────────────────────────────────────────────
CALIBRATION_SPLIT = 0.1    # fraction of summaries used to fit the Fisher diagonal
PRIOR_PRECISION   = 1.0    # lambda: higher = narrower posterior
SAMPLE_COUNT      = 20     # posterior samples per summary during scoring

# ── HuggingFace Hub ───────────────────────────────────────────────────────────
HF_REPO_ID    = "rdisipio/summarizer-uncertainty-models"
HF_SUBFOLDER  = "bart-large-xsum-lora-laplace"

print("Config OK")

## 2. Load data

In [ ]:
import json
import random

random.seed(SEED)

pairs = []
with SUMMARIES_FILE.open() as f:
    for line in f:
        r = json.loads(line)
        src = r.get("paragraph_text", "").strip()
        tgt = r.get("summary", "").strip()
        if src and tgt:
            pairs.append({"source": src, "target": tgt})

print(f"Loaded {len(pairs)} (source, summary) pairs")

random.shuffle(pairs)
val_size   = max(1, int(len(pairs) * VAL_SPLIT))
val_pairs  = pairs[:val_size]
train_pairs = pairs[val_size:]
print(f"Train: {len(train_pairs)}  |  Val: {len(val_pairs)}")

## 3. Fine-tune with LoRA

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Loading base model: {BASE_MODEL}")
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize(batch):
    inputs = tokenizer(batch["source"], max_length=MAX_SOURCE_LENGTH, truncation=True, padding=False)
    labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq]
        for seq in labels["input_ids"]
    ]
    return inputs

train_dataset = Dataset.from_list(train_pairs).map(tokenize, batched=True, remove_columns=["source", "target"])
val_dataset   = Dataset.from_list(val_pairs).map(tokenize, batched=True, remove_columns=["source", "target"])

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=False,
    bf16=BF16,
    dataloader_num_workers=0,
    seed=SEED,
    logging_steps=10,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

print(f"Saving adapter to {ADAPTER_DIR}")
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Training complete.")

## 4. Fit the diagonal Laplace approximation

In [ ]:
from peft import PeftModel
from src.lora_laplace_backend import (
    LoraLaplaceBackend, fit_laplace_approximation, save_laplace_sampler
)

# Partition calibration data from the full pair list
all_pairs_raw = []
with SUMMARIES_FILE.open() as f:
    for line in f:
        r = json.loads(line)
        src = r.get("paragraph_text", "").strip()
        tgt = r.get("summary", "").strip()
        if src and tgt:
            all_pairs_raw.append((src, tgt))

random.seed(SEED)
random.shuffle(all_pairs_raw)
cal_size = max(1, int(len(all_pairs_raw) * CALIBRATION_SPLIT))
calibration_data = all_pairs_raw[:cal_size]
print(f"Calibration samples: {len(calibration_data)}")

# Load the saved adapter
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

base  = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
peft_model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=True)
tok   = AutoTokenizer.from_pretrained(BASE_MODEL)

backend = LoraLaplaceBackend(peft_model=peft_model, tokenizer=tok, device=device)

print("Fitting Laplace approximation...")
sampler = fit_laplace_approximation(
    backend=backend,
    calibration_data=calibration_data,
    prior_precision=PRIOR_PRECISION,
)

save_laplace_sampler(sampler, str(SAMPLER_FILE))
print(f"Sampler saved to {SAMPLER_FILE}")

## 5. Score the calibration corpus

In [ ]:
from collections import defaultdict
from tqdm.notebook import tqdm
from src.scorer import SummaryUncertaintyScorer

scorer = SummaryUncertaintyScorer(backend=backend, posterior_sampler=sampler)

# Group remaining records by base chunk ID
scoring_pairs = all_pairs_raw[cal_size:]

all_sentence_scores = []
skipped = 0

with SCORES_FILE.open("w") as out:
    for source, summary in tqdm(scoring_pairs, desc="Scoring"):
        try:
            result = scorer.score_summary(
                source=source, summary=summary,
                sample_count=SAMPLE_COUNT, seed=SEED,
            )
        except Exception as e:
            print(f"[ERROR] {e}")
            skipped += 1
            continue

        sentence_scores = [
            {"sentence_index": s.sentence_index, "sentence_text": s.sentence_text,
             "uncertainty": s.uncertainty}
            for s in result.sentence_results
        ]
        all_sentence_scores.extend(sentence_scores)

        if sentence_scores:
            avg = sum(s["uncertainty"] for s in sentence_scores) / len(sentence_scores)
            out.write(json.dumps({"sentence_scores": sentence_scores, "uncertainty_avg": avg},
                                 ensure_ascii=False) + "\n")

print(f"Scored {len(scoring_pairs) - skipped} summaries, {len(all_sentence_scores)} sentences. Skipped: {skipped}")

## 6. Fit quantile normalizer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

scores_array = np.array([s["uncertainty"] for s in all_sentence_scores])

print(f"Sentences scored : {len(scores_array)}")
print(f"Min    : {scores_array.min():.6f}")
print(f"p25    : {np.percentile(scores_array, 25):.6f}")
print(f"Median : {np.median(scores_array):.6f}")
print(f"p75    : {np.percentile(scores_array, 75):.6f}")
print(f"Max    : {scores_array.max():.6f}")

plt.figure(figsize=(8, 3))
plt.hist(scores_array, bins=60, color="steelblue", edgecolor="white")
plt.xlabel("Epistemic MI (uncertainty)")
plt.ylabel("Sentences")
plt.title("Sentence-level uncertainty distribution (LoRA-Laplace)")
plt.tight_layout()
plt.show()

In [ ]:
QUANTILE_POINTS = [0.0, 0.25, 0.5, 0.75, 1.0]
boundaries = [float(np.quantile(scores_array, q)) for q in QUANTILE_POINTS]

print("Quantile boundaries:", [f"{b:.6f}" for b in boundaries])

CONFIG_DIR.mkdir(exist_ok=True)
QUANTILES_FILE.write_text(json.dumps({"boundaries": boundaries}, indent=2) + "\n")
print(f"Written to {QUANTILES_FILE}")

## 7. Upload to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)
print(f"Repo ready: {HF_REPO_ID}")

files_to_upload = [
    (ADAPTER_DIR / "adapter_config.json",       f"{HF_SUBFOLDER}/adapter_config.json"),
    (ADAPTER_DIR / "adapter_model.safetensors", f"{HF_SUBFOLDER}/adapter_model.safetensors"),
    (ADAPTER_DIR / "tokenizer.json",            f"{HF_SUBFOLDER}/tokenizer.json"),
    (ADAPTER_DIR / "tokenizer_config.json",     f"{HF_SUBFOLDER}/tokenizer_config.json"),
    (SAMPLER_FILE,                              f"{HF_SUBFOLDER}/laplace_sampler.npz"),
    (QUANTILES_FILE,                            f"{HF_SUBFOLDER}/uncertainty_quantiles_lora_laplace.json"),
]

for local, remote in files_to_upload:
    print(f"Uploading {local.name} → {remote}")
    api.upload_file(
        path_or_fileobj=str(local),
        path_in_repo=remote,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"\nAll files uploaded to https://huggingface.co/{HF_REPO_ID}")